# Exercise 1 - Pick a new CME event, prepare it, and launch a 200-run HUXt batch

**Time budget:** ~20 min hands-on, then it runs in the background &nbsp;|&nbsp; **Builds on:** notebooks 02 + 04

The committed surrogate is for **2017-09-06**. In this exercise you choose a *different*,
contrasting event from `data_dir/events.csv`, generate its HUXt inner-boundary (the WSA+
checkpoint is already downloaded, so only the GONG magnetogram is fetched), and launch the
`design -> run` batch that produces its `results.csv`.

**A 200-run batch takes far longer than the session** (~1-2 min/run). So: **launch it now and
let it run** while you do Exercises 2 and 3 on the committed 2017-09-06 data. `run_design`
writes `results.csv` incrementally, so Exercise 4 can analyse however many runs have finished.

**By the end you can:** turn a row of `events.csv` into a HUXt boundary and a running surrogate
batch, and reason about which event makes an interesting contrast to 2017-09-06.

## Running on Google Colab

A few things differ from a local run:

- **Bootstrap restart.** The first cell installs HUXt + WSA+ and may restart the runtime once. When it reconnects, **re-run the first cell** (or *Runtime ▸ Run all*).
- **No Terminal on the free tier.** Where the steps say "run it in a terminal", use the **background-launch cell in Task 1.3** instead - `subprocess.Popen(...)` returns immediately, so the kernel stays free while the batch runs.
- **Each notebook is its own VM.** Opening Exercises 2-4 as separate Colab tabs gives each a *separate* filesystem, so they will **not** see this notebook's `results.csv`. Exercises 2-3 only use the committed 2017-09-06 data (fine alone), but **do Exercise 4 in this same tab** - or mount Google Drive and write results there:
  ```python
  from google.colab import drive; drive.mount("/content/drive")
  # then set GP_ROOT = Path("/content/drive/MyDrive/conecast_runs/gp_surrogate")
  ```
- **Disconnects.** Colab recycles the VM after ~90 min idle and a background run dies with it (unsaved `results.csv` is lost unless on Drive). Keep this tab active, and consider **lowering the batch to `n=60-100`** in the design cell so it finishes within the session (keep 200 as homework).

In [ ]:
# --- Google Colab bootstrap (no-op locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "/content/conecast"
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    try:
        import sunpy, huxt, wsaplus  # noqa: F401
        print("Colab bootstrap complete; cwd =", os.getcwd())
    except ModuleNotFoundError:
        print("Installing sunpy + WSA+ + HUXt (one-time, ~2 min)...")
        os.system("pip install -q sunpy wsaplus")
        os.system("pip install -q "
                  "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt'")
        print("Done - restarting the runtime. When it reconnects, RUN THIS CELL AGAIN.")
        os.kill(os.getpid(), 9)
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
BASE_DIR = cwd if (cwd / "scripts").exists() else cwd.parent
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
print("BASE_DIR =", BASE_DIR)

In [ ]:
import pandas as pd, numpy as np
import gp_huxt_surrogate as gp

events = pd.read_csv(BASE_DIR / "data_dir" / "events.csv")
display(events[["event", "longitude", "latitude", "width", "speed", "notes"]])

## Task 1.1 - Choose a contrasting event

2017-09-06 is a wide (~88 deg), fast (~1238 km/s) CME launched near Earth's longitude - it
mostly **hits**. Pick something that should behave differently, e.g.:

- `2017-09-10` - extreme west-limb (longitude 85 deg), very fast (2650 km/s): expect many
  **misses** -> a harder, more balanced classifier problem.
- `2012-07-12` - very wide (140 deg), Earth-directed: expect even more hits than 2017-09-06.

**TODO:** set `EVENT` to your choice and predict, before running anything, whether its batch
will be hit-heavy or miss-heavy, and why.

In [ ]:
EVENT = ...  # TODO: your chosen event id (e.g. "2017-09-10") from the table above
row = events.loc[events["event"] == EVENT].iloc[0]
print(row[["event", "longitude", "latitude", "width", "speed", "notes"]])
# Your prediction (hit-heavy / miss-heavy + one-line reason):

## Task 1.2 - Generate the HUXt boundary for your event

`generate_huxt_input.py` runs the GONG -> WSA+ -> sub-Earth track -> boundary pipeline and
writes `data_dir/sw/<event>/` (boundary + `event_config.yaml`). WSA+ is cached locally; the
GONG magnetogram for the event date is downloaded.

**TODO:** fill the event id. (Skip this cell if the boundary already exists.)

In [ ]:
import subprocess
boundary = BASE_DIR / "data_dir" / "sw" / EVENT / f"v_boundary_{EVENT}.npz"
if boundary.exists():
    print("boundary already present:", boundary)
else:
    cmd = [sys.executable, str(SCRIPT_DIR / "generate_huxt_input.py"), "--event", EVENT]  # TODO: confirm EVENT
    print("running:", " ".join(cmd))
    print(subprocess.run(cmd, cwd=BASE_DIR, capture_output=True, text=True).stdout[-2000:])

## Task 1.3 - Build the design and launch the 200-run batch

`make_design` lays down a 200-point Latin hypercube around the seed; `run_design` runs HUXt at
each point and appends a row to `results.csv`. **Launch the run in a terminal so it keeps going
in the background** while you do Exercises 2-3:

```bash
python scripts/gp_huxt_surrogate.py --event <EVENT> design --n 200
python scripts/gp_huxt_surrogate.py --event <EVENT> run        # leave this running
```

**TODO:** create the design from here (fast), then start `run` in a terminal.

In [ ]:
DATA_ROOT = BASE_DIR / "data_dir" / "sw"
GP_ROOT   = BASE_DIR / "runs" / "gp_surrogate"
DEFAULT_SPAN = dict(inject_hour=1.0, longitude=30.0, latitude=20.0, width=40.0, speed_fraction=0.25)

# TODO: build the 200-point design for EVENT (reuse the script defaults from DEFAULT_SPAN).
gp.make_design(EVENT, DATA_ROOT, GP_ROOT, n=200, seed=42, force=True,
               span_inject_hour=DEFAULT_SPAN["inject_hour"],
               span_longitude=DEFAULT_SPAN["longitude"],
               span_latitude=DEFAULT_SPAN["latitude"],
               span_width=DEFAULT_SPAN["width"],
               span_speed_fraction=DEFAULT_SPAN["speed_fraction"])
print("design written. Launch the batch with the cell below, then move on to Exercise 2.")

### Launch the run

Locally you can run this in a terminal and leave it going:

```bash
python scripts/gp_huxt_surrogate.py --event <EVENT> run
```

Anywhere - and the only option on Colab - launch it as a **background process** from the cell
below. It returns immediately, so move on to Exercise 2 and re-run the *progress* cell now and
then to watch `results.csv` grow.

In [ ]:
import subprocess
(GP_ROOT / EVENT).mkdir(parents=True, exist_ok=True)
log_path = GP_ROOT / EVENT / "run.log"
proc = subprocess.Popen(
    [sys.executable, str(SCRIPT_DIR / "gp_huxt_surrogate.py"), "--event", EVENT, "run"],
    cwd=str(BASE_DIR), stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
)
print(f"launched `run` in the background (PID {proc.pid}); logging to {log_path}")

In [ ]:
# Re-run this any time to check progress (run_design appends a row per completed HUXt sample).
results_path = GP_ROOT / EVENT / "results.csv"
n_rows = (sum(1 for _ in open(results_path)) - 1) if results_path.exists() else 0
print(f"{EVENT}: {n_rows} completed rows so far")
if log_path.exists():
    print("--- last log lines ---")
    print(log_path.read_text()[-400:])

**Note your prediction (Task 1.1) here**, then go to Exercise 2. You will return to *your*
`results.csv` in Exercise 4 and analyse whatever has completed.

### Stretch
- Run two contrasting events at once (e.g. a limb event and a central wide event) and compare
  their hit rates in Exercise 4.